### Read the Bronze tables

In [0]:
from pyspark.sql import functions as F

BRONZE_PRODUCTS = "olist_mlops.bronze.products"
BRONZE_TRANSLATIONS = "olist_mlops.bronze.category_translation"

SILVER_PRODUCTS = "olist_mlops.silver.products_clean"
SILVER_TRANSLATIONS = "olist_mlops.silver.category_translation_clean"

bronze_products = spark.table(BRONZE_PRODUCTS)
bronze_translations = spark.table(BRONZE_TRANSLATIONS)

print("Products:", bronze_products.count())
print("Translations:", bronze_translations.count())

bronze_products.printSchema()

### Normalize category names

In [0]:
translations_clean = (
    bronze_translations

    .withColumn(
        "product_category_name",
        F.lower(F.trim(F.col("product_category_name")))
    )

    .withColumn(
        "product_category_name_english",
        F.trim(F.col("product_category_name_english"))
    )

    .withColumn(
        "silver_processed_at",
        F.current_timestamp()
    )
)

# Validate source row count
assert translations_clean.count() == 71

# Validate category keys
invalid_keys = translations_clean.filter(
    F.col("product_category_name").isNull()
    | (F.col("product_category_name") == "")
).count()

duplicate_keys = (
    translations_clean
    .groupBy("product_category_name")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Invalid category keys:", invalid_keys)
print("Duplicate category keys:", duplicate_keys)

assert invalid_keys == 0
assert duplicate_keys == 0

print("CATEGORY TRANSLATION VALIDATION PASSED")

### Write the translation table

In [0]:
(
    translations_clean.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TRANSLATIONS)
)

assert spark.table(SILVER_TRANSLATIONS).count() == 71

print("Created:", SILVER_TRANSLATIONS)

### Rename and cast numerical columns

In [0]:
products_typed = (
    bronze_products

    .withColumnRenamed(
        "product_name_lenght",
        "product_name_length"
    )

    .withColumnRenamed(
        "product_description_lenght",
        "product_description_length"
    )

    .withColumn(
        "product_category_name",
        F.lower(F.trim(F.col("product_category_name")))
    )
)

numeric_columns = [
    "product_name_length",
    "product_description_length",
    "product_photos_qty",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

for column in numeric_columns:

    products_typed = products_typed.withColumn(
        column,
        F.expr(f"try_cast(`{column}` AS INT)")
    )

products_typed.printSchema()

### Check casting failures

In [0]:
source_numeric_columns = {
    "product_name_lenght": "product_name_length",
    "product_description_lenght": "product_description_length",
    "product_photos_qty": "product_photos_qty",
    "product_weight_g": "product_weight_g",
    "product_length_cm": "product_length_cm",
    "product_height_cm": "product_height_cm",
    "product_width_cm": "product_width_cm"
}

for source_col, target_col in source_numeric_columns.items():

    failed_casts = bronze_products.filter(
        F.col(source_col).isNotNull()
        & (F.trim(F.col(source_col)) != "")
        & F.expr(
            f"try_cast(`{source_col}` AS INT)"
        ).isNull()
    ).count()

    print(f"{target_col}: {failed_casts} failed conversions")

    assert failed_casts == 0

print("NUMERIC CAST VALIDATION PASSED")

### Enrich product records

In [0]:
category_lookup = (
    spark.table(SILVER_TRANSLATIONS)
    .select(
        F.col("product_category_name").alias("lookup_category"),
        F.col("product_category_name_english")
    )
)

products_clean = (
    products_typed

    .join(
        category_lookup,
        products_typed.product_category_name
            == category_lookup.lookup_category,
        "left"
    )

    .drop("lookup_category")

    .withColumn(
        "category_translation_status",
        F.when(
            F.col("product_category_name").isNull()
            | (F.col("product_category_name") == ""),
            F.lit("MISSING_CATEGORY")
        )
        .when(
            F.col("product_category_name_english").isNull(),
            F.lit("UNMAPPED")
        )
        .otherwise(F.lit("TRANSLATED"))
    )

    .withColumn(
        "silver_processed_at",
        F.current_timestamp()
    )
)

display(
    products_clean.select(
        "product_id",
        "product_category_name",
        "product_category_name_english",
        "category_translation_status"
    ).limit(10)
)

### Validate counts and unique keys

In [0]:
source_count = bronze_products.count()
silver_count = products_clean.count()

distinct_products = (
    products_clean
    .select("product_id")
    .distinct()
    .count()
)

null_product_ids = products_clean.filter(
    F.col("product_id").isNull()
).count()

print("Bronze products:", source_count)
print("Silver products:", silver_count)
print("Distinct product IDs:", distinct_products)
print("Null product IDs:", null_product_ids)

assert source_count == silver_count == 32951
assert distinct_products == 32951
assert null_product_ids == 0

print("PRODUCT ROW-COUNT VALIDATION PASSED")

### Validate translation coverage

In [0]:
display(
    products_clean
    .groupBy("category_translation_status")
    .count()
    .orderBy("category_translation_status")
)

### Validate the expected distribution

In [0]:
status_counts = {
    row["category_translation_status"]: row["count"]
    for row in (
        products_clean
        .groupBy("category_translation_status")
        .count()
        .collect()
    )
}

assert status_counts.get("TRANSLATED", 0) == 32328
assert status_counts.get("UNMAPPED", 0) == 13
assert status_counts.get("MISSING_CATEGORY", 0) == 610

print("CATEGORY COVERAGE VALIDATION PASSED")

### Save as Delta

In [0]:
(
    products_clean.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_PRODUCTS)
)

assert spark.table(SILVER_PRODUCTS).count() == 32951

print("Created:", SILVER_PRODUCTS)

### Verify everything using SQL

In [0]:
%sql

SELECT
    category_translation_status,
    COUNT(*) AS total_products
FROM olist_mlops.silver.products_clean
GROUP BY category_translation_status
ORDER BY category_translation_status;

### Now inspect missing numerical features

In [0]:
%sql

SELECT
    COUNT(*) AS total_products,
    COUNT_IF(product_weight_g IS NULL) AS missing_weight,
    COUNT_IF(product_length_cm IS NULL) AS missing_length,
    COUNT_IF(product_height_cm IS NULL) AS missing_height,
    COUNT_IF(product_width_cm IS NULL) AS missing_width,
    COUNT_IF(product_photos_qty IS NULL) AS missing_photos
FROM olist_mlops.silver.products_clean;

### check for unexpected non-positive physical measurements

In [0]:
%sql

SELECT
    COUNT_IF(product_weight_g <= 0) AS non_positive_weight,
    COUNT_IF(product_length_cm <= 0) AS non_positive_length,
    COUNT_IF(product_height_cm <= 0) AS non_positive_height,
    COUNT_IF(product_width_cm <= 0) AS non_positive_width
FROM olist_mlops.silver.products_clean;

### Handle invalid product weights

In [0]:
%sql

SELECT
    product_id,
    product_category_name,
    product_weight_g,
    product_length_cm,
    product_height_cm,
    product_width_cm
FROM olist_mlops.silver.products_clean
WHERE product_weight_g <= 0
ORDER BY product_weight_g;

In [0]:
from pyspark.sql import functions as F

products_clean = (
    products_clean

    .withColumn(
        "product_weight_g_clean",
        F.when(
            F.col("product_weight_g") > 0,
            F.col("product_weight_g")
        ).otherwise(F.lit(None).cast("int"))
    )

    .withColumn(
        "product_weight_quality",
        F.when(
            F.col("product_weight_g").isNull(),
            "MISSING"
        )
        .when(
            F.col("product_weight_g") <= 0,
            "NON_POSITIVE"
        )
        .otherwise("VALID")
    )
)

display(
    products_clean
    .groupBy("product_weight_quality")
    .count()
)